# Test the model published on Hugging Face

Loads **your model straight from the Hugging Face Hub**. **Google Drive is not used at all**: nothing is mounted, read or written there. The notebook downloads the model, checks that the upload is complete and sound, and generates Stories from the downloaded files only. It does **no training** and uploads nothing. Run the cells top to bottom.

- Put your repo in `HF_REPO_ID` (cell 1). It is left blank in the committed file on purpose.
- A **private** repo needs a Colab secret named `HF_TOKEN` (key icon in the left sidebar, a token with **read** access, "Notebook access" switched on). A public repo needs no token.
- The model code (`dlm/`) comes from the public GitHub repo; the weights, tokenizer and config come from the Hub.
- A GPU is not required (the model is small), but **Runtime -> Change runtime type -> GPU** makes it faster.
- Cell 1 has `QUIT_WHEN_DONE = True`: the last cell ends the session so no credits are spent while you read. Set it to `False` to keep experimenting.

In [ ]:
# 1. Settings (no Google Drive: everything comes from the Hugging Face Hub and GitHub)
HF_REPO_ID = ""           # the model repo to test, e.g. "your-username/dlm-tinystories-36m_cosine" (type yours here)
SCHEDULE = "cosine"       # the noise schedule the model was trained with ("linear" or "cosine"); generation uses the same one
QUIT_WHEN_DONE = True     # the LAST cell ends the Colab session (nothing needs saving: the model lives on the Hub)
GRACE_SECONDS = 60        # countdown before the session ends (stop the last cell during it to cancel)

In [ ]:
# 2. Get the latest code from GitHub (public repo; tokenizer/ is committed there). Re-running this cell is safe.
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone -q $REPO /content/dlm_repo
%cd /content/dlm_repo
!git log --oneline -1

In [ ]:
# 3. Dependencies: install ONLY what is missing (never numpy / torch: reinstalling them can break Colab's live kernel)
import importlib.util, subprocess, sys
missing = [p for p in ("tokenizers", "safetensors", "huggingface_hub") if importlib.util.find_spec(p) is None]
print("missing:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def sh(*args):
    """Run `python run.py <args>` and print its output."""
    p = subprocess.run([sys.executable, "run.py", *args], capture_output=True, text=True)
    print(p.stdout + p.stderr)
    return p.returncode

In [ ]:
# 4. Download the model from the Hub and check the upload is complete. CKPT = the downloaded folder: every later cell uses ONLY these
# files (model.safetensors, config.json, tokenizer.json), exactly what a stranger who downloads your repo would get.
import json, os
if not HF_REPO_ID:
    raise SystemExit("Set HF_REPO_ID in cell 1 (for example 'your-username/dlm-tinystories-36m_cosine'), then run cells 1 and 4 again.")
from dlm.hub import HubError, download_snapshot
try:
    CKPT = download_snapshot(HF_REPO_ID)
except HubError as e:
    raise SystemExit(f"{e}\nA private repo needs a Colab secret named HF_TOKEN (read access, 'Notebook access' on). "
                     "Also check the repo name for typos.")
print("Downloaded to", CKPT, "\n")
files = sorted((os.path.relpath(os.path.join(d, f), CKPT), os.path.getsize(os.path.join(d, f)))
               for d, _, fs in os.walk(CKPT) for f in fs if not d.split(os.sep)[-1].startswith("."))
for name, size in files:
    print(f"  {name:32s} {size / 1e6:8.2f} MB")
names = {n for n, _ in files}
need = ("config.json", "model.safetensors", "tokenizer.json", "generation_config.json", "README.md")
absent = [n for n in need if n not in names]
print("\nAll required files are on the Hub." if not absent else f"\nMISSING from the Hub repo: {absent}")
if os.path.isfile(f"{CKPT}/run_info.json"):
    info = json.load(open(f"{CKPT}/run_info.json"))
    r = info.get("recipe", {})
    print(f"Trained for {r.get('steps')} steps, noise schedule {r.get('schedule')!r}; stopped_at_step = {info.get('stopped_at_step')}")
    if r.get("schedule") and r["schedule"] != SCHEDULE:
        print(f"WARNING: the model was trained with the {r['schedule']!r} schedule but SCHEDULE = {SCHEDULE!r} in cell 1")
if os.path.isfile(f"{CKPT}/README.md") and "intermediate checkpoint" in open(f"{CKPT}/README.md").read().lower():
    print("NOTE: the model card says this is an intermediate checkpoint (it was exported from resume.pt before the run ended).")
gen_cfg = json.load(open(f"{CKPT}/generation_config.json")) if "generation_config.json" in names else {}
if gen_cfg.get("schedule") not in (None, SCHEDULE):
    print(f"NOTE: the repo's generation_config.json says schedule = {gen_cfg['schedule']!r}; the sampler here ignores that file "
          f"and uses SCHEDULE = {SCHEDULE!r} from cell 1.")

In [ ]:
# 5. What the Hub page shows: the training curves and the evaluation report that were uploaded with the model.
from IPython.display import Image, display
if os.path.isfile(f"{CKPT}/training_curves.png"):
    display(Image(f"{CKPT}/training_curves.png"))
else:
    print("No training_curves.png in the Hub repo.")
if os.path.isfile(f"{CKPT}/eval/report.md"):
    print(open(f"{CKPT}/eval/report.md").read())
else:
    print("No eval/report.md in the Hub repo.")

In [ ]:
# 6. Is the downloaded model sound? Loads ONLY the Hub files and checks: every weight is finite, input and output embeddings are tied,
# the tokenizer matches the model's vocabulary, no [MASK] is left in generated text, and greedy decoding (temperature 0) repeats exactly.
import torch
from dlm.sample import generate_text, load_checkpoint
model, tok = load_checkpoint(CKPT, "cpu")
cfg = model.config
n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {n_params:,} parameters | {cfg.n_layers} layers x width {cfg.d_model} | vocab {cfg.vocab_size:,} | max length {cfg.seq_len}")
problems = []
if not all(torch.isfinite(p).all() for p in model.parameters()):
    problems.append("non-finite weights")
if model.lm_head.weight is not model.tok_emb.weight:
    problems.append("input and output embeddings are not tied")
if tok.get_vocab_size() != cfg.vocab_size:
    problems.append(f"tokenizer has {tok.get_vocab_size()} tokens, the model {cfg.vocab_size}")
if tok.token_to_id("[MASK]") != cfg.mask_id or tok.token_to_id("[PAD]") != cfg.pad_id:
    problems.append("the tokenizer's [MASK]/[PAD] ids differ from the model's")
for prompt in ("", "Once upon a time, there was a little"):
    kw = dict(prompt=prompt, max_new_tokens=96, schedule=SCHEDULE)
    try:
        a = generate_text(model, tok, seed=1, temperature=0.0, **kw)
        b = generate_text(model, tok, seed=2, temperature=0.0, **kw)
    except Exception as e:  # e.g. NaN weights: report it instead of stopping
        problems.append(f"generation failed for prompt {prompt!r} ({type(e).__name__}: {str(e)[:80]})")
        continue
    if a["text"] != b["text"]:
        problems.append(f"greedy decoding is not deterministic (prompt {prompt!r})")
    if cfg.mask_id in a["ids"][a["prefix_len"]:a["covered"]]:
        problems.append(f"a [MASK] token is left in the generated text (prompt {prompt!r})")
    if not a["text"].strip():
        problems.append(f"empty text (prompt {prompt!r})")
print("\nRESULT:", "the Hub model loads, generates and greedy decoding repeats exactly." if not problems else "PROBLEMS: " + "; ".join(problems))

In [ ]:
# 7. Three Stories from scratch (no prompt), generated from the downloaded Hub files. Each line below the Story is its length and whether it finished.
sh("generate", "--ckpt", CKPT, "--n", "3", "--seed", "0", "--schedule", SCHEDULE)

In [ ]:
# 8. Continue YOUR opening words (edit the form and re-run as often as you like). --trace shows the unmasking as text rows
# (every row = one step; blocks of 32 are filled left to right, most confident first). Avoid double quotes in the prompt.
PROMPT = "Once upon a time, there was a little"  #@param {type:"string"}
HOW_MANY = 2  #@param {type:"integer"}
SEED = 0  #@param {type:"integer"}
TEMPERATURE = 0.8  #@param {type:"number"}
import glob, os
from IPython.display import Image, display
for f in glob.glob("/content/unmask*.gif"):
    os.remove(f)
sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", str(HOW_MANY), "--seed", str(SEED),
   "--temperature", str(TEMPERATURE), "--schedule", SCHEDULE, "--trace", "--gif", "/content/unmask.gif")
gifs = sorted(glob.glob("/content/unmask*.gif"))
if gifs:
    display(Image(gifs[0]))   # the unmasking animation of the first sample

In [ ]:
# 9. Does decoding speed or randomness change quality? Same opening words: greedy (temperature 0, deterministic), the default,
# and a fast setting (60 steps for ~480 tokens = about 8 tokens per step). Compare the text and the printed time.
import time
for label, extra in (("greedy", ["--temperature", "0"]), ("default", []), ("fast, 60 steps", ["--steps", "60"])):
    print(f"\n===== {label} =====")
    t0 = time.time()
    sh("generate", "--ckpt", CKPT, "--prompt", PROMPT, "--n", "1", "--seed", "1", "--schedule", SCHEDULE, *extra)
    print(f"[{label}: {time.time() - t0:.1f} s including start-up]")

In [ ]:
# 10. END THE SESSION. Nothing needs saving (the model is on the Hub and Drive is not used). The GIF in cell 8 is only shown on screen.
try:
    from dlm.colab import finish_session
    finish_session(None, enabled=QUIT_WHEN_DONE, grace_seconds=GRACE_SECONDS)
except Exception as e:  # never end the session by accident
    print(f"NOT ending the session ({type(e).__name__}: {e}). Stop it yourself: Runtime -> Disconnect and delete runtime.")